# local_parquet_demo — Pipeline Walkthrough

**Zero-credential quickstart — CSV in, gold Parquet out.**

| Step | Layer | What happens |
|------|-------|--------------|
| 1 | Bronze | `seed.py` reads `orders.csv` → writes `ORDERS.parquet` |
| 2 | Silver | Rename columns, cast types, flag large orders via UDF |
| 3 | Gold | Aggregate by customer and by status |

Run cells top to bottom.

## Setup — navigate to example root

In [1]:
import os, sys
from pathlib import Path

# Two levels up: ipynb/ → demo/ → local_parquet_demo/
example_root = Path(os.getcwd()).parent.parent
os.chdir(example_root)
print(f'Working directory: {os.getcwd()}')

Working directory: /workspace/openmedallion/examples/local_parquet_demo


## Step 1 — Bronze: seed from CSV

`seed.py` reads `data/source/orders.csv` and writes `data/bronze/ORDERS.parquet`.

In [2]:
import polars as pl

src  = Path('data/source/orders.csv')
dest = Path('data/bronze')
dest.mkdir(parents=True, exist_ok=True)

df = pl.read_csv(src)
df.write_parquet(dest / 'ORDERS.parquet')
print(f'✅  Seeded {len(df)} rows → {dest / "ORDERS.parquet"}')
print()
print('Source data preview:')
print(df.head(5))

✅  Seeded 15 rows → data/bronze/ORDERS.parquet

Source data preview:
shape: (5, 5)
┌──────────┬─────────────┬───────────────┬────────┬───────────┐
│ ORDER_ID ┆ CUSTOMER_ID ┆ CUSTOMER_NAME ┆ AMOUNT ┆ STATUS    │
│ ---      ┆ ---         ┆ ---           ┆ ---    ┆ ---       │
│ i64      ┆ i64         ┆ str           ┆ f64    ┆ str       │
╞══════════╪═════════════╪═══════════════╪════════╪═══════════╡
│ 1        ┆ 101         ┆ Alice         ┆ 250.0  ┆ completed │
│ 2        ┆ 102         ┆ Bob           ┆ 45.5   ┆ completed │
│ 3        ┆ 103         ┆ Charlie       ┆ 130.0  ┆ completed │
│ 4        ┆ 101         ┆ Alice         ┆ 80.0   ┆ pending   │
│ 5        ┆ 102         ┆ Bob           ┆ 180.0  ┆ completed │
└──────────┴─────────────┴───────────────┴────────┴───────────┘


## Step 2 — Silver: rename, cast, UDF

`backend/silver.yaml` applies three transforms in sequence:

| Transform | What it does |
|-----------|-------------|
| `rename` | `ORDER_ID` → `order_id`, `CUSTOMER_ID` → `customer_id`, etc. |
| `cast` | `order_id` → `Int64`, `customer_id` → `Int64`, `amount` → `Float64` |
| `udf` | `flag_large_orders(df, threshold=100.0)` → adds `is_large_order` |


In [3]:
!medallion run demo --layer silver


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  medallion  ·  demo  ·  bronze → silver
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋  [config] bronze: demo/backend/bronze.yaml
📋  [config] silver: demo/backend/silver.yaml
📋  [config] gold  : demo/backend/gold.yaml

  ⏭️  bronze  skipped (existing files)

── Silver ─────────────────────────────────────────────────

********************************************************************************
> silver [openmedallion.pipeline.nodes.silver()] encountered an error          <
> Node inputs:
{'bronze': "{'ORDERS': PosixPath('data/bronze/ORDERS.parquet')...",
 'config': "{'pipeline': {'name': 'demo'}, 'paths': {'bronze':..."}
********************************************************************************
Traceback (most recent call last):
  File "/opt/venv/lib/python3.12/site-packages/hamilton/execution/graph_functions.py", line 321, in execute_lifecycle_for_node
    result = __node_(**__kwargs)
             ^^

In [ ]:
silver = pl.read_parquet('data/silver/orders.parquet')
print(f'Silver: {silver.shape[0]} rows, {silver.shape[1]} columns')
print(silver.head(8))
print(f'\nLarge orders (amount >= 100): {silver["is_large_order"].sum()} of {len(silver)}')

## Step 3 — Gold: aggregate

`backend/gold.yaml` declares two aggregations:

- **`orders_by_customer`** — group by `customer_id, customer_name` → `total_orders`, `total_spent`
- **`orders_by_status`** — group by `status` → `num_orders`, `total_amount`

In [ ]:
!medallion run demo --layer gold

In [ ]:
gold_dir = Path('data/gold/demo')

print('── orders_by_customer ──')
by_customer = pl.read_parquet(gold_dir / 'orders_by_customer.parquet')
print(by_customer.sort('total_spent', descending=True))

print('\n── orders_by_status ──')
by_status = pl.read_parquet(gold_dir / 'orders_by_status.parquet')
print(by_status.sort('total_amount', descending=True))

total = by_customer['total_spent'].sum()
print(f'\nGrand total revenue: ${total:,.2f}')

## Things to Try

- **Change the UDF threshold**: edit `backend/silver.yaml` → `args.threshold: 200.0`, then re-run the silver cell
- **Add a metric**: add `{column: amount, agg: mean, alias: avg_order}` to `backend/gold.yaml`, re-run gold
- **Inspect the DAG**: run `!medallion dag` in a new cell